# Nuclear Waste Canister Temperature Prediction — Random Forest
**CIVIL-226 - Introduction to Machine Learning for Engineers**

**Members:** NAJA Nour | CLERICI Alessandro

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)
print('Imports OK')

## 2. Chargement des données

In [ ]:
sensors = pd.read_parquet('data/sensors.parquet')
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')

n_before = len(sensors)
sensors  = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)
print(f'Doublons supprimés : {n_before - len(sensors)} (N206, N213)')
print(f'Sensors : {sensors.shape}  ->  {sensors.columns.tolist()}')
print(f'Train   : {train.shape}   ->  {train.columns.tolist()}')
print(f'Test    : {test.shape}    ->  {test.columns.tolist()}')

## 3. Exploration des données

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['temperature'].dropna(), bins=50, color='steelblue',
             edgecolor='white', range=(-200, 200))
axes[0].set_title('Distribution des températures')
axes[0].set_xlabel('Température')
axes[0].set_ylabel('Count')

temp_by_time = train.groupby('time')['temperature'].mean()
axes[1].plot(temp_by_time.index / 1e6, temp_by_time.values, color='tomato', linewidth=0.8)
axes[1].set_title('Température moyenne au cours du temps')
axes[1].set_xlabel('Temps (x10⁶ s)')
axes[1].set_ylabel('Température moyenne')

plt.tight_layout()
plt.show()

# Positions des capteurs
train_sensors = set(train['sensor'].unique())
test_sensors  = set(test['sensor'].unique())
s_train = sensors[sensors['sensor'].isin(train_sensors)]
s_test  = sensors[sensors['sensor'].isin(test_sensors)]

plt.figure(figsize=(8, 4))
plt.scatter(s_train['coor_x'], s_train['coor_y'], c='steelblue', s=20, alpha=0.6, label='Train')
plt.scatter(s_test['coor_x'],  s_test['coor_y'],  c='tomato',    s=40, alpha=0.9, marker='^', label='Test')
plt.axvline(1.4, color='gray', linestyle='--', alpha=0.5, label='Buffer/OPA')
plt.xlabel('coor_x [m]'); plt.ylabel('coor_y [m]')
plt.title('Positions des capteurs (2D)')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Preprocessing

### 4.1 Suppression des NaN

In [ ]:
print(f'Lignes avant nettoyage : {len(train)}')
train_clean = train.dropna(subset=['temperature']).copy()
print(f'Lignes après suppression NaN : {len(train_clean)}')
print(f'Supprimées : {len(train) - len(train_clean)}')

### 4.2 Feature Engineering

On joint les coordonnées spatiales et on ajoute des features physiquement motivées :
- `dist_center` : distance à l'origine
- `dist_canister` : distance au centre du canister (x=0.7, y=1.2)
- `is_opa` : zone OPA (x > 1.4m) vs buffer
- `time_norm` : temps normalisé [0, 1]
- `time_log` : log(1+t) pour capturer la dynamique logarithmique de diffusion

In [ ]:
def add_features(df, sensors_df):
    """
    Joint les coordonnées et ajoute des features dérivées.
    
    Features:
    - coor_x, coor_y    : position spatiale
    - dist_center       : distance à l'origine
    - dist_canister     : distance au canister (x=0.7, y=1.2)
    - is_opa            : 1 si zone OPA (x > 1.4m), 0 si buffer
    - time_norm         : temps normalisé [0, 1]
    - time_log          : log(1+t)
    """
    merged = df.copy()
    if 'coor_x' not in merged.columns:
        merged = merged.merge(sensors_df, on='sensor', how='left')

    t_max = train_clean['time'].max()

    merged['dist_center']   = np.sqrt(merged['coor_x']**2 + merged['coor_y']**2)
    merged['dist_canister'] = np.sqrt((merged['coor_x'] - 0.7)**2 + (merged['coor_y'] - 1.2)**2)
    merged['is_opa']        = (merged['coor_x'] > 1.4).astype(float)
    merged['time_norm']     = merged['time'] / t_max
    merged['time_log']      = np.log1p(merged['time'])

    return merged

train_feat = add_features(train_clean, sensors)
test_feat  = add_features(test, sensors)

print('Features :', train_feat.columns.tolist())
display(train_feat.head(3))

In [ ]:
TARGET   = 'temperature'
FEATURES = ['coor_x', 'coor_y', 'time_norm', 'time_log', 'power',
            'dist_center', 'dist_canister', 'is_opa']

X      = train_feat[FEATURES].values
y      = train_feat[TARGET].values
X_test = test_feat[FEATURES].values

print(f'X shape      : {X.shape}')
print(f'X_test shape : {X_test.shape}')

### 4.3 Filtrage des outliers

On filtre les températures physiquement impossibles avant le split.

In [ ]:
mask_all = (y > -10) & (y < 200)
print(f'Aberrants supprimés : {(~mask_all).sum()} sur {len(y)}')

X_clean = X[mask_all]
y_clean = y[mask_all]

### 4.4 Split train/validation & Normalisation

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

assert len(X_test_s) == len(test)
print(f'Train : {X_train_s.shape} | Val : {X_val_s.shape} | Test : {X_test_s.shape}')

## 5. Modèle — Random Forest

Le Random Forest entraîne T arbres de décision indépendants sur des sous-ensembles des données et moyenne leurs prédictions :

$$\hat{y} = \frac{1}{T}\sum_{t=1}^{T} h_t(\mathbf{x})$$

Cette approche réduit la variance par rapport à un seul arbre et capture les non-linéarités entre position, temps et puissance.

In [ ]:
# Subsample pour accélérer l'entraînement
np.random.seed(42)
idx = np.random.choice(len(X_train_s), size=200_000, replace=False)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_s[idx], y_train[idx])
print('Entraînement terminé.')

## 6. Évaluation

In [ ]:
y_pred_val_rf = rf.predict(X_val_s)
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_val_rf))
print(f'Random Forest — RMSE validation : {rmse_rf:.4f}')

In [ ]:
# True vs predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(y_val, y_pred_val_rf, s=2, alpha=0.2)
mn, mx = min(y_val.min(), y_pred_val_rf.min()), max(y_val.max(), y_pred_val_rf.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('True temperature [°C]')
plt.ylabel('Predicted temperature [°C]')
plt.title(f'Random Forest — True vs Predicted (RMSE={rmse_rf:.2f}°C)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(6, 4), color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. Prédictions finales & Soumission

In [ ]:
y_pred = rf.predict(X_test_s)

submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})

assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'Random Forest — RMSE validation : {rmse_rf:.4f}')
print(f'submission.csv sauvegardé — {len(submission)} lignes')
display(submission.head())

In [ ]:
# Sanity check : train vs test predictions over time
test_with_pred = test.copy()
test_with_pred['temperature_pred'] = y_pred

temp_train = train_clean.groupby('time')['temperature'].mean()
temp_pred  = test_with_pred.groupby('time')['temperature_pred'].mean()

plt.figure(figsize=(10, 4))
plt.plot(temp_train.index / 1e6, temp_train.values,
         label='Train (vraies)', color='tomato', alpha=0.7, linewidth=0.8)
plt.plot(temp_pred.index / 1e6, temp_pred.values,
         label='Test (prédites)', color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Comparaison train vs prédictions')
plt.xlabel('Temps (x10⁶ s)')
plt.ylabel('Température moyenne')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Pistes d'amélioration

- Tuning des hyperparamètres (`max_depth`, `min_samples_leaf`, `n_estimators`)
- Séparation Buffer/OPA (deux modèles spécialisés)
- Séparation en 3 reps selon le profil de puissance
- Features supplémentaires (`cum_energy`, `power/dist²`)
- Gradient Boosting (XGBoost, LightGBM)